## Test
Här gör jag en väldigt simpel test model. Jag vill testa embeddings och några grundläggande frågor.
Några viktiga insikter som jag tar med mig är att omfattande chunking förmodligen inte kommer vara nödvändigt (tillägg två veckor senare är att jag hade OTROLIGT fel :D ). Datan är redan semantiskt uppdelad med avgränasade lektioner inom kurser. Att dela upp dessa riskerar att försämra modellens förmåga att förstå helheten. Jag vet dock att det finns lektioner som är betydligt större och som kommer behöva delas upp. 

En annan insikt är hur begränsad Geminis gratisnivå är med dess limits vilket innebär att antalet anrop kommer att bli ett problem vid embeddings av flera kurser.

In [ ]:
import os 
from google import genai
from dotenv import load_dotenv
from pypdf import PdfReader
from google.genai import types
import json
import numpy as np

In [10]:
load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [ ]:
folder_path = r"C:\YA BI analyst\Kurser\BI25M AI & IoT\Kunskapskontroll 1\AI-IoT-Kunskapskrav-1\data\cleaned\3064200"

def load_course(folder_path):
    documents = []

    for file in os.listdir(folder_path):
        if not file.endswith(".json"):
            continue

        path = os.path.join(folder_path, file)

        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        text = data.get("text", "").strip()

        # Filtrera bort skräp
        if len(text) < 20 or len(text.split()) < 5:
            continue

        documents.append({
            "text": text,
            "metadata": {
                "course_id": data.get("course_id"),
                "course_name": data.get("course_name"),
                "chapter_title": data.get("chapter_title"),
                "lesson_title": data.get("lesson_title"),
                "file": file
            }
        })

    return documents

Kollar så att det laddats korrekt. Där en lektion är ett objekt i listan

In [4]:
for i, doc in enumerate(documents):
    print(f"\n--- Dokument {i+1} ---")
    print("Fil:", doc["metadata"]["file"])
    print("Längd:", len(doc["text"]))


--- Dokument 1 ---
Fil: 63949181.json
Längd: 1482

--- Dokument 2 ---
Fil: 63949184.json
Längd: 96

--- Dokument 3 ---
Fil: 63949186.json
Längd: 1358

--- Dokument 4 ---
Fil: 73644264.json
Längd: 788

--- Dokument 5 ---
Fil: 73644304.json
Längd: 2022


In [ ]:
def create_embeddings(text, model="gemini-embedding-001", task_type="SEMANTIC_SIMILARITY"):
    return client.models.embed_content(
        model=model,
        contents=text,
        config=types.EmbedContentConfig(task_type=task_type)
    )

def embed_documents(documents):
    embedded_docs = []

    for doc in documents:
        response = create_embeddings(doc["text"])
        embedding = response.embeddings[0].values

        embedded_docs.append({
            "text": doc["text"],
            "embedding": embedding,
            "metadata": doc["metadata"]
        })

    return embedded_docs

In [13]:
embedded_docs = embed_documents(documents)

print(f"Antal embeddings: {len(embedded_docs)}")
print(f"Dimension på första embedding: {len(embedded_docs[0]['embedding'])}")
print(embedded_docs[0]["metadata"])

Antal embeddings: 5
Dimension på första embedding: 3072
{'course_id': '3064200', 'course_name': 'Segling för Nybörjare', 'chapter_title': 'Båtdelar och Terminologi', 'lesson_title': 'Seglingsspråk', 'file': '63949181.json'}


funktion för cosine similarity

In [ ]:
def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

funktion för att söka fram bästa chunks

In [15]:
def search_documents(query, embedded_docs, top_k=3):
    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents=query,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY")
    )
    
    query_embedding = response.embeddings[0].values

    scored_docs = []

    for doc in embedded_docs:
        score = cosine_similarity(query_embedding, doc["embedding"])
        scored_docs.append({
            "score": score,
            "text": doc["text"],
            "metadata": doc["metadata"]
        })

    scored_docs = sorted(scored_docs, key=lambda x: x["score"], reverse=True)

    return scored_docs[:top_k]

testa bara retrieval först

In [16]:
results = search_documents("Vad betyder att reva?", embedded_docs, top_k=3)

for i, result in enumerate(results, start=1):
    print(f"\n--- Träff {i} ---")
    print("Score:", result["score"])
    print("Lesson:", result["metadata"]["lesson_title"])
    print("Fil:", result["metadata"]["file"])
    print("Text:")
    print(result["text"])


--- Träff 1 ---
Score: 0.673940862600146
Lesson: Seglingsspråk
Fil: 63949181.json
Text:
Lär dig vanliga termer och uttryck som används inom segling för att kommunicera effektivt på vattnet.
Att förstå den specifika terminologi som används för att beskriva olika delar av en båt är avgörande för effektiv kommunikation mellan seglare. Låt oss utforska några viktiga seglingstermer.
Reva
: Processen att minska storleken på ett segel för att anpassa sig till förändrade vindförhållanden.
Slå
: Att ändra en segelbåts riktning genom att vrida fören mot vinden.
Gippa
: Att ändra en segelbåts riktning genom att vrida aktern genom vinden.
Lovart:
Där vinden kommer från
Lä.
Motsats mot lovart
Bog
: båtens riktning i förhållande till vinden, ex kryssbog.
Bära av:
att mer handkraft förhindra att en båt törnar emot ett annat föremål. På våra båtar används en fender, gärna kulfendern för detta. Vi får aldrig ha en hand eller fot mellan vår båt och annat föremål, ex en annan båt och kaj.
Sätta segel:
A

bygg ett enkelt RAG-svar

In [17]:
def ask_rag(query, embedded_docs, top_k=3):
    results = search_documents(query, embedded_docs, top_k=top_k)

    context = "\n\n".join(
        [f"Lektion: {r['metadata']['lesson_title']}\n{r['text']}" for r in results]
    )

    prompt = f"""
Du är en hjälpsam assistent för en seglingskurs.

Använd endast information från kontexten nedan.
Om svaret inte finns i kontexten, säg att du inte vet.

KONTEXT:
{context}

FRÅGA:
{query}
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text, results

testa modellen

In [28]:
answer, results = ask_rag("Hur sätter man segel? Förklara så utförligt som du kan", embedded_docs, top_k=3)

print("SVAR:")
print(answer)

SVAR:
Att sätta segel innebär att hissa upp seglet och skota hem.


In [24]:
answer, results = ask_rag("Hur fungerar en dieselmotor på båten?", embedded_docs, top_k=3)

print("SVAR:")
print(answer)

SVAR:
Jag vet inte hur en dieselmotor fungerar på båten utifrån den information som gavs.
